# ENIGH 2024 — Construcción del dataset para predecir el ingreso mensual personal

**Proyecto:** `ml_predict_income`
**Fuente:** INEGI, Encuesta Nacional de Ingresos y Gastos de los Hogares (ENIGH) 2024, *Nueva Serie*.
**Objetivo del dataset:** una fila por **persona adulta** con su **ingreso monetario mensual (MXN)** y los
atributos demográficos, educativos, geográficos, ocupacionales y de contexto del hogar que alimentarán al modelo.

---

## What this notebook does

| Step | Output |
|---|---|
| 1 | Setup: paths, readers, inventory of the raw ENIGH tables |
| 2 | **Target**: monthly monetary income per person, from `ingresos` |
| 3 | Person block: demographics + education, from `poblacion` |
| 4 | Job block: main-job characteristics, from `trabajos` |
| 5 | Geography block, from `viviendas` |
| 6 | Household-context block (income-free), from `hogares` + `concentradohogar` + `viviendas` |
| 7 | Join everything at person level |
| 8 | Apply the analysis universe (adults 18+, income > 0) and keep the flags |
| 9 | Quality checks & validation against INEGI aggregates |
| 10 | Save `data/processed/enigh2024_ingreso_personas.{parquet,csv}` + data dictionary |

> The narrative is in English; **column names stay in Spanish** so they map 1:1 to the ENIGH
> dictionaries (`diccionario_de_datos/`) and catalogs (`catalogos/`) shipped with the microdata.


## Design decisions (and why)

### 1. Target = monthly **monetary** income, all sources

ENIGH records income at the **person × income-key × quarter** level in the `ingresos` table (`ing_tri`).
We keep every key INEGI classifies as *current income* and drop `P050`–`P066`, the
**"percepciones financieras y de capital"** — loans received, withdrawals from savings, sales of assets,
inheritances, insurance payouts. Those are balance-sheet movements, not income, and INEGI reports them
separately (`percep_tot` in `concentradohogar`).

The target is `ingreso_mensual = ing_tri_monetario / 3`.

**Why monetary only:** ENIGH's headline `ing_cor` also includes *non-monetary* income (imputed rent
`estim_alqu`, in-kind pay, self-consumption, gifts in kind). Those components are **only measured at the
household level** — they cannot be assigned to a person. A person-level target must therefore be the
monetary part. This is verified numerically in §9.

| Component built | ENIGH keys |
|---|---|
| `ing_mens_sueldos_salarios` | P001–P022 — wages, piecework, overtime, bonuses, tips, profit-sharing, 13th-month pay, secondary job |
| `ing_mens_negocios` | P068–P081 — self-employment / business profits by activity |
| `ing_mens_trabajo_menores` | P067 — work income of under-12s |
| `ing_mens_rentas` | P023–P031 — rent of land and property, interest, royalties |
| `ing_mens_transferencias` | P032–P048, P101–P108 — pensions, scholarships, remittances, social programs, donations |
| `ing_mens_otros` | P049 |
| **excluded** | **P050–P066** — financial perceptions, *not* income |

### 2. Universe = adults 18+ with income > 0

Rows are persons aged **18 or more** whose monthly monetary income is **strictly positive** — the classic
earnings-equation population, a clean target with no zero inflation. The unfiltered table stays in memory
as `personas`, with the flags `es_adulto`, `tiene_ingreso` and `ocupado`, so the other two universes
(all adults including zeros / only people who worked) can be rebuilt without re-running the pipeline.

### 3. Leakage control

No income or expenditure variable from `concentradohogar` enters the feature set (`ing_cor`, `ingtrab`,
`sueldos`, `gasto_mon`, …). The household block is restricted to **composition, dwelling and assets**.
Household assets (car, internet, computer) are correlated with income almost by construction; they are kept
but grouped behind the `INCLUIR_ACTIVOS_HOGAR` switch so they can be dropped with one flag.

### 4. Survey design is carried, not applied

`factor` (expansion weight), `upm` (primary sampling unit) and `est_dis` (design stratum) travel with every
row. They are **not** features. Use `factor` for any population-level statistic, and `upm` as the grouping
key when splitting train/test so the same cluster never lands on both sides.

### 5. Missing-value convention

ENIGH codes *not specified* as `&` and *not applicable* as an empty field. Both are read as `NaN`.
For job variables, "not applicable because the person does not work" is later encoded as the explicit
category `"NO_APLICA"` — that is information, not a missing value.

### 6. Prices

ENIGH 2024 amounts are already expressed by INEGI in **August 2024 pesos**. No deflation is applied here.


---
## 1. Setup

Requirements: `pandas >= 2.0`, `numpy`, `pyarrow` (for parquet).

```bash
pip install pandas numpy pyarrow
```


In [1]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.6 | numpy 2.4.6


In [2]:
# --------------------------------------------------------------- Parámetros
EDAD_MINIMA = 18               # universo: personas de 18 años o más
SOLO_INGRESO_POSITIVO = True   # universo: ingreso monetario mensual > 0
INCLUIR_ACTIVOS_HOGAR = True   # activos del hogar (auto, internet, computadora, ...)
GUARDAR_CSV = True             # además del parquet
ANIO = 2024

NOMBRE_SALIDA = "enigh2024_ingreso_personas"

# --------------------------------------------------------------- Rutas
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "conjunto_de_datos_enigh2024_ns_csv"
PROC_DIR = PROJECT_ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DIR.exists(), f"No se encontró la carpeta de microdatos: {RAW_DIR}"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROC_DIR     :", PROC_DIR)

PROJECT_ROOT : d:\repositories\projects\mlops\ml_predict_income
RAW_DIR      : d:\repositories\projects\mlops\ml_predict_income\data\raw\conjunto_de_datos_enigh2024_ns_csv
PROC_DIR     : d:\repositories\projects\mlops\ml_predict_income\data\processed


In [3]:
# --------------------------------------------------------------- Lectores
ID_PERSONA = ["folioviv", "foliohog", "numren"]
ID_HOGAR = ["folioviv", "foliohog"]
ID_VIVIENDA = ["folioviv"]

# ENIGH: "&" = no especificado, campo vacío = no aplica. Ambos -> NaN.
NA_ENIGH = {"": np.nan, "&": np.nan, "NA": np.nan}


def ruta_tabla(tabla: str) -> Path:
    # Ruta al CSV de una tabla ENIGH (p. ej. "poblacion").
    carpeta = RAW_DIR / f"conjunto_de_datos_{tabla}_enigh{ANIO}_ns"
    return carpeta / "conjunto_de_datos" / f"conjunto_de_datos_{tabla}_enigh{ANIO}_ns.csv"


def leer_enigh(tabla: str, usecols=None) -> pd.DataFrame:
    # Lee una tabla ENIGH como texto (preserva ceros a la izquierda) y normaliza los NA.
    df = pd.read_csv(ruta_tabla(tabla), usecols=usecols, dtype=str, low_memory=False)
    for col in df.columns:
        df[col] = df[col].str.strip().replace(NA_ENIGH)
    return df


def leer_catalogo(tabla: str, nombre: str) -> pd.DataFrame:
    # Lee un catálogo de códigos (carpeta catalogos/) de una tabla ENIGH.
    ruta = RAW_DIR / f"conjunto_de_datos_{tabla}_enigh{ANIO}_ns" / "catalogos" / f"{nombre}.csv"
    cat = pd.read_csv(ruta, dtype=str)
    for col in cat.columns:
        cat[col] = cat[col].str.strip()
    return cat


def mapa_catalogo(tabla: str, nombre: str) -> dict:
    # Catálogo de 2 columnas -> dict {codigo: descripcion}.
    cat = leer_catalogo(tabla, nombre)
    return dict(zip(cat.iloc[:, 0], cat.iloc[:, 1]))


def num(serie: pd.Series) -> pd.Series:
    return pd.to_numeric(serie, errors="coerce")


def cuantiles_ponderados(valores, pesos, qs=(0.10, 0.25, 0.50, 0.75, 0.90, 0.99)) -> dict:
    # Cuantiles con factor de expansión.
    v = np.asarray(valores, dtype=float)
    w = np.asarray(pesos, dtype=float)
    orden = np.argsort(v)
    v, w = v[orden], w[orden]
    acum = np.cumsum(w) / w.sum()
    return {f"p{q * 100:g}": float(np.interp(q, acum, v)) for q in qs}

In [4]:
# Inventario de las tablas disponibles
tablas = sorted(p.name.replace("conjunto_de_datos_", "").replace(f"_enigh{ANIO}_ns", "")
                for p in RAW_DIR.iterdir() if p.is_dir())
inventario = pd.DataFrame(
    [{"tabla": t,
      "csv": ruta_tabla(t).exists(),
      "MB": round(ruta_tabla(t).stat().st_size / 1e6, 1) if ruta_tabla(t).exists() else np.nan}
     for t in tablas]
).sort_values("MB", ascending=False, ignore_index=True)
inventario

,tabla,csv,MB
0,gastoshogar,True,579.00
1,poblacion,True,92.80
2,concentradohogar,True,45.60
3,ingresos,True,35.20
4,gastospersona,True,32.40
5,hogares,True,25.50
6,trabajos,True,20.10
7,viviendas,True,16.10
8,noagroimportes,True,9.70
9,noagro,True,6.80


---
## 2. Target — monthly monetary income per person

The `ingresos` table is *long*: one row per **person × income key × quarter**, with `ing_tri` = the amount
received over the reference quarter. We classify every key, drop the non-income ones, pivot to one row per
person, and divide by 3.


In [5]:
# Grupos de claves de ingreso (catálogo ingresos_cat.csv)
GRUPOS_CLAVE = {
    "sueldos_salarios":  list(range(1, 23)),                        # P001-P022
    "negocios":          list(range(68, 82)),                       # P068-P081
    "trabajo_menores":   [67],                                      # P067
    "rentas":            list(range(23, 32)),                       # P023-P031
    "transferencias":    list(range(32, 49)) + list(range(101, 109)),  # P032-P048, P101-P108
    "otros":             [49],                                      # P049
}
# P050-P066: percepciones financieras y de capital -> NO son ingreso corriente
CLAVES_NO_INGRESO = list(range(50, 67))

CLAVE_A_GRUPO = {c: g for g, claves in GRUPOS_CLAVE.items() for c in claves}
assert not (set(CLAVE_A_GRUPO) & set(CLAVES_NO_INGRESO)), "Un grupo se traslapa con las claves excluidas"

catalogo_ingresos = mapa_catalogo("ingresos", "ingresos_cat")
print(f"{len(catalogo_ingresos)} claves de ingreso en el catálogo")
print(f"{len(CLAVE_A_GRUPO)} clasificadas como ingreso | {len(CLAVES_NO_INGRESO)} excluidas")

83 claves de ingreso en el catálogo
72 clasificadas como ingreso | 17 excluidas


In [6]:
ingresos = leer_enigh("ingresos", usecols=ID_PERSONA + ["clave", "ing_tri"])
ingresos["ing_tri"] = num(ingresos["ing_tri"]).fillna(0.0)
ingresos["clave_num"] = ingresos["clave"].str.extract(r"P(\d+)")[0].astype(int)
ingresos["grupo"] = ingresos["clave_num"].map(CLAVE_A_GRUPO)

# Auditoría: ninguna clave debe quedar sin clasificar
sin_clasificar = ingresos.loc[ingresos["grupo"].isna() & ~ingresos["clave_num"].isin(CLAVES_NO_INGRESO), "clave"].unique()
assert len(sin_clasificar) == 0, f"Claves sin clasificar: {sin_clasificar}"

print(f"registros de ingreso : {len(ingresos):,}")
print(f"personas con registro: {ingresos.groupby(ID_PERSONA).ngroups:,}")
ingresos.head()

registros de ingreso : 391,563
personas con registro: 202,365


,folioviv,foliohog,numren,clave,ing_tri,clave_num,grupo
0,0100001901,1,01,P001,"48,952.17",1,sueldos_salarios
1,0100001901,1,01,P004,"17,608.69",4,sueldos_salarios
2,0100001901,1,01,P006,"4,402.17",6,sueldos_salarios
3,0100001901,1,01,P008,"4,402.17",8,sueldos_salarios
4,0100001901,1,01,P009,"3,423.91",9,sueldos_salarios


In [7]:
# Aportación de cada grupo al total nacional (trimestral, sin ponderar)
auditoria_claves = (
    ingresos.assign(grupo=ingresos["grupo"].fillna("EXCLUIDO (P050-P066)"))
    .groupby("grupo")
    .agg(registros=("ing_tri", "size"), monto_trimestral=("ing_tri", "sum"))
    .sort_values("monto_trimestral", ascending=False)
)
auditoria_claves["% del ingreso"] = (
    100 * auditoria_claves["monto_trimestral"]
    / auditoria_claves.loc[auditoria_claves.index != "EXCLUIDO (P050-P066)", "monto_trimestral"].sum()
).round(2)
auditoria_claves

,registros,monto_trimestral,% del ingreso
grupo,,,
sueldos_salarios,238275,"3,911,199,793.06",70.95
transferencias,90452,"980,772,729.53",17.79
negocios,34423,"551,049,755.94",10.00
EXCLUIDO (P050-P066),21273,"222,902,727.60",4.04
rentas,4040,"62,570,414.84",1.14
otros,1768,"6,005,088.75",0.11
trabajo_menores,1332,"711,378.00",0.01


In [8]:
# Pivote a nivel persona: una columna por grupo + total
target = (
    ingresos.dropna(subset=["grupo"])
    .pivot_table(index=ID_PERSONA, columns="grupo", values="ing_tri", aggfunc="sum")
    .fillna(0.0)
)
target.columns.name = None
target = target.reindex(columns=list(GRUPOS_CLAVE), fill_value=0.0)

# Trimestral -> mensual
target = (target / 3).round(2)
target.columns = [f"ing_mens_{c}" for c in target.columns]
target["ingreso_mensual"] = target.sum(axis=1).round(2)

# Derivados útiles para el modelado
target["ingreso_mensual_laboral"] = (
    target["ing_mens_sueldos_salarios"] + target["ing_mens_negocios"] + target["ing_mens_trabajo_menores"]
).round(2)

target = target.reset_index()
print(f"personas con algún ingreso monetario: {len(target):,}")
target.head()

personas con algún ingreso monetario: 200,474


,folioviv,foliohog,numren,ing_mens_sueldos_salarios,ing_mens_negocios,ing_mens_trabajo_menores,ing_mens_rentas,ing_mens_transferencias,ing_mens_otros,ingreso_mensual,ingreso_mensual_laboral
0,0100001901,1,01,"26,263.04",0.00,0.00,0.00,0.00,0.00,"26,263.04","26,263.04"
1,0100001901,1,02,"10,385.86",0.00,0.00,0.00,0.00,0.00,"10,385.86","10,385.86"
2,0100001902,1,01,"18,913.04",0.00,0.00,0.00,0.00,0.00,"18,913.04","18,913.04"
3,0100001902,1,03,"9,782.61",0.00,0.00,0.00,0.00,0.00,"9,782.61","9,782.61"
4,0100001904,1,01,"3,953.80",0.00,0.00,0.00,0.00,0.00,"3,953.80","3,953.80"


In [9]:
target.filter(like="ing_mens_").describe().T[["count", "mean", "50%", "max"]]

,count,mean,50%,max
ing_mens_sueldos_salarios,"200,474.00","6,503.25","3,375.00","5,673,913.04"
ing_mens_negocios,"200,474.00",916.24,0.00,"729,344.26"
ing_mens_trabajo_menores,"200,474.00",1.18,0.00,"3,000.00"
ing_mens_rentas,"200,474.00",104.04,0.00,"344,262.29"
ing_mens_transferencias,"200,474.00","1,630.76",0.00,"299,409.83"
ing_mens_otros,"200,474.00",9.98,0.00,"245,901.64"


---
## 3. Person block — demographics and education (`poblacion`)

308,598 rows, one per household member. We keep identity, demographics, education, ethnicity, disability
and labour-force status, and we derive:

- **`anios_escolaridad`** — ENIGH does not ship years of schooling; it is reconstructed from
  `nivelaprob` (level attained) × `gradoaprob` (grade within the level), using `antec_esc` to resolve the
  ambiguous levels (technical/commercial studies and *normal*, whose length depends on the prior level).
  This follows the CONEVAL convention used for the educational-lag indicator.
- **`nivel_educativo`** — a 6-way collapse of `nivelaprob`.
- **`n_dificultades` / `tiene_discapacidad`** — from the eight `disc_*` items, counting only
  *"lo hace con mucha dificultad"* (3) and *"no puede hacerlo"* (4), the WG-SS severity cut-off.
- **`situacion_conyugal`**, **`parentesco_grupo`**, **`condicion_actividad`**.


In [10]:
COLS_POB = ID_PERSONA + [
    "parentesco", "sexo", "edad", "pais_nac", "afrod", "hablaind", "etnia", "alfabetism",
    "asis_esc", "nivelaprob", "gradoaprob", "antec_esc", "edo_conyug", "segsoc",
    "trabajo_mp", "num_trabaj", "act_pnea1", "hijos_sob",
    "disc_ver", "disc_oir", "disc_brazo", "disc_camin", "disc_apren", "disc_vest",
    "disc_habla", "disc_acti",
    "entidad", "est_dis", "upm", "factor",
]
pob = leer_enigh("poblacion", usecols=COLS_POB)
print(f"poblacion: {len(pob):,} filas")
assert not pob.duplicated(ID_PERSONA).any(), "Hay personas duplicadas en poblacion"
pob.head(3)

poblacion: 308,598 filas


,folioviv,foliohog,numren,parentesco,sexo,edad,pais_nac,afrod,disc_ver,disc_oir,disc_brazo,disc_camin,disc_apren,disc_vest,disc_habla,disc_acti,hablaind,etnia,alfabetism,asis_esc,nivelaprob,gradoaprob,antec_esc,edo_conyug,segsoc,hijos_sob,trabajo_mp,act_pnea1,num_trabaj,entidad,est_dis,upm,factor
0,0100001901,1,01,101,1,32,1,2,1,1,1,1,1,1,1,1,2,2,1,2,03,3,NaN,3,1,NaN,1,NaN,1,01,001,0000001,207
1,0100001901,1,02,201,2,24,1,2,1,1,1,1,1,1,1,1,2,2,1,2,04,3,NaN,3,2,2,1,NaN,1,01,001,0000001,207
2,0100001901,1,03,301,2,5,1,2,1,1,1,1,1,1,1,1,2,2,1,1,01,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01,001,0000001,207


In [11]:
# ---------------------------------------------- Años de escolaridad (CONEVAL)
# Años acumulados ANTES de iniciar cada nivel
BASE_NIVEL = {
    "00": 0,   # ninguno
    "01": 0,   # preescolar
    "02": 0,   # primaria        -> 0 + grado (1-6)
    "03": 6,   # secundaria      -> 6 + grado (1-3)
    "04": 9,   # preparatoria    -> 9 + grado (1-3)
    "07": 12,  # licenciatura    -> 12 + grado
    "08": 16,  # especialidad    -> 16 + grado
    "09": 16,  # maestría        -> 16 + grado
    "10": 18,  # doctorado       -> 18 + grado
}
# Niveles cuya duración depende del antecedente escolar (antec_esc)
BASE_ANTECEDENTE = {"1": 6, "2": 9, "3": 12, "4": 16, "5": 18}


def calcular_anios_escolaridad(nivel, grado, antecedente):
    if pd.isna(nivel):
        return np.nan
    g = 0 if pd.isna(grado) else int(grado)
    if nivel in ("00", "01"):
        return 0
    if nivel in BASE_NIVEL:
        return min(BASE_NIVEL[nivel] + g, 24)
    if nivel == "05":                       # normal
        return min(BASE_ANTECEDENTE.get(antecedente, 9) + g, 24)
    if nivel == "06":                       # estudios técnicos o comerciales
        return min(BASE_ANTECEDENTE.get(antecedente, 9) + g, 24)
    return np.nan


NIVEL_EDUCATIVO = {
    "00": "01 Sin instrucción", "01": "01 Sin instrucción",
    "02": "02 Primaria",
    "03": "03 Secundaria",
    "04": "04 Media superior", "05": "04 Media superior", "06": "04 Media superior",
    "07": "05 Licenciatura",
    "08": "06 Posgrado", "09": "06 Posgrado", "10": "06 Posgrado",
}
SITUACION_CONYUGAL = {
    "1": "Unión libre", "2": "Casado(a)", "3": "Casado(a)", "4": "Casado(a)",
    "5": "Separado(a)", "6": "Divorciado(a)", "7": "Viudo(a)", "8": "Soltero(a)",
}
PARENTESCO_GRUPO = {
    "1": "Jefe(a)", "2": "Cónyuge", "3": "Hijo(a)", "4": "Trabajador(a) doméstico(a)",
    "5": "Sin parentesco", "6": "Otro pariente", "7": "Huésped", "9": "No especificado",
}
ACT_PNEA = {
    "1": "Buscó trabajo", "2": "Pensionado(a)/jubilado(a)", "3": "Quehaceres del hogar",
    "4": "Estudia", "5": "Limitación permanente", "6": "Otra situación",
}
DISC_COLS = ["disc_ver", "disc_oir", "disc_brazo", "disc_camin",
             "disc_apren", "disc_vest", "disc_habla", "disc_acti"]

In [12]:
personas = pd.DataFrame(index=pob.index)

# --- identificadores y diseño muestral
personas[ID_PERSONA] = pob[ID_PERSONA]
personas["entidad"] = pob["entidad"]
personas["est_dis"] = pob["est_dis"]
personas["upm"] = pob["upm"]
personas["factor"] = num(pob["factor"])

# --- demografía
personas["edad"] = num(pob["edad"])
personas["sexo"] = pob["sexo"].map({"1": "Hombre", "2": "Mujer"})
personas["es_mujer"] = (pob["sexo"] == "2").astype("int8")
personas["parentesco_grupo"] = pob["parentesco"].str[0].map(PARENTESCO_GRUPO)
personas["es_jefe_hogar"] = (pob["parentesco"] == "101").astype("int8")
personas["situacion_conyugal"] = pob["edo_conyug"].map(SITUACION_CONYUGAL)
personas["vive_en_pareja"] = pob["edo_conyug"].isin(["1", "2", "3", "4"]).astype("int8")
personas["lugar_nacimiento"] = pob["pais_nac"].map(
    {"1": "Misma entidad", "2": "Otra entidad", "3": "Estados Unidos", "4": "Otro país"})
personas["es_migrante_interno"] = pob["pais_nac"].isin(["2", "3", "4"]).astype("int8")
personas["hijos_sobrevivientes"] = num(pob["hijos_sob"])

# --- pertenencia étnica
personas["habla_lengua_indigena"] = (pob["hablaind"] == "1").astype("int8")
personas["se_considera_indigena"] = (pob["etnia"] == "1").astype("int8")
personas["es_afrodescendiente"] = (pob["afrod"] == "1").astype("int8")

# --- educación
personas["nivel_educativo"] = pob["nivelaprob"].map(NIVEL_EDUCATIVO)
personas["anios_escolaridad"] = [
    calcular_anios_escolaridad(n, g, a)
    for n, g, a in zip(pob["nivelaprob"], pob["gradoaprob"], pob["antec_esc"])
]
personas["sabe_leer_escribir"] = (pob["alfabetism"] == "1").astype("int8")
personas["asiste_a_escuela"] = (pob["asis_esc"] == "1").astype("int8")

# --- discapacidad (corte de severidad: mucha dificultad / no puede hacerlo)
personas["n_dificultades"] = pob[DISC_COLS].isin(["3", "4"]).sum(axis=1).astype("int8")
personas["tiene_discapacidad"] = (personas["n_dificultades"] > 0).astype("int8")

# --- condición de actividad
personas["trabajo_mes_pasado"] = (pob["trabajo_mp"] == "1").astype("int8")
personas["n_trabajos"] = num(pob["num_trabaj"]).fillna(0).astype("int8")
personas["actividad_no_ocupado"] = pob["act_pnea1"].map(ACT_PNEA)
personas["cotiza_seguridad_social"] = pob["segsoc"].map({"1": 1, "2": 0}).astype("Int8")

print(f"personas: {len(personas):,} filas × {personas.shape[1]} columnas")
personas.head(3)

personas: 308,598 filas × 30 columnas


,folioviv,foliohog,numren,entidad,est_dis,upm,factor,edad,sexo,es_mujer,parentesco_grupo,es_jefe_hogar,situacion_conyugal,vive_en_pareja,lugar_nacimiento,es_migrante_interno,hijos_sobrevivientes,habla_lengua_indigena,se_considera_indigena,es_afrodescendiente,nivel_educativo,anios_escolaridad,sabe_leer_escribir,asiste_a_escuela,n_dificultades,tiene_discapacidad,trabajo_mes_pasado,n_trabajos,actividad_no_ocupado,cotiza_seguridad_social
0,0100001901,1,01,01,001,0000001,207,32,Hombre,0,Jefe(a),1,Casado(a),1,Misma entidad,0,NaN,0,0,0,03 Secundaria,9.00,1,0,0,0,1,1,NaN,1
1,0100001901,1,02,01,001,0000001,207,24,Mujer,1,Cónyuge,0,Casado(a),1,Misma entidad,0,2.00,0,0,0,04 Media superior,12.00,1,0,0,0,1,1,NaN,0
2,0100001901,1,03,01,001,0000001,207,5,Mujer,1,Hijo(a),0,NaN,0,Misma entidad,0,NaN,0,0,0,01 Sin instrucción,0.00,1,1,0,0,0,0,NaN,<NA>


In [13]:
# Validación de la escolaridad derivada
chk = pd.crosstab(pob["nivelaprob"].map(NIVEL_EDUCATIVO), personas["anios_escolaridad"].round(0))
display(chk)
print("NaN en anios_escolaridad:", personas["anios_escolaridad"].isna().sum(),
      f"({personas['anios_escolaridad'].isna().mean():.1%}) — corresponden a menores sin nivel declarado")
print("escolaridad media adultos 18+ (ponderada): "
      f"{np.average(personas.loc[personas.edad >= 18, 'anios_escolaridad'].fillna(0), weights=personas.loc[personas.edad >= 18, 'factor']):.2f} años")

anios_escolaridad,0.00,1.00,2.00,3.00,4.00,5.00,6.00,7.00,8.00,9.00,10.00,11.00,12.00,13.00,14.00,15.00,16.00,17.00,18.00,19.00,20.00,21.00,22.00,23.00,24.00
nivelaprob,,,,,,,,,,,,,,,,,,,,,,,,,
01 Sin instrucción,29051,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
02 Primaria,0,8404,10473,13327,8870,8367,37005,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
03 Secundaria,0,0,0,0,0,0,0,7603,9240,62967,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
04 Media superior,0,0,0,0,0,0,0,35,101,233,7614,9071,40448,645,546,1012,469,104,0,0,0,0,0,0,0
05 Licenciatura,0,0,0,0,0,0,0,0,0,0,0,0,0,3948,3905,4965,13053,12866,0,0,0,0,0,0,0
06 Posgrado,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,400,1922,624,352,192,102,33,6


NaN en anios_escolaridad: 10645 (3.4%) — corresponden a menores sin nivel declarado
escolaridad media adultos 18+ (ponderada): 10.06 años


---
## 4. Job block — main job (`trabajos`)

One row per **person × job** (`id_trabajo` = 1 main, 2 secondary). We keep the **main job** and add a flag
for having a secondary one. 150,382 people report a main job, so these columns are `NaN` for the rest —
pensioners, remittance recipients, rentiers. That absence is meaningful, so at the end of §7 the categorical
job columns are filled with the explicit category `"NO_APLICA"`.

Derived variables:

- **`posicion_ocupacion`** — employee / employer / own-account / unpaid worker, from `subor`, `indep`,
  `personal` and `pago`.
- **`sinco_grupo`** — 1-digit SINCO-2011 major occupational group (10 classes); `sinco` (4 digits) is kept too.
- **`scian_sector`** — 2-digit SCIAN economic sector; `scian` (4 digits) is kept too.
- **`n_prestaciones`** — count of the 19 fringe benefits (`pres_1`…`pres_19`).
- **`empleo_formal`** — proxy for formality: the job provides access to a public health institution
  (IMSS, ISSSTE, ISSSTE estatal, PEMEX/Defensa/Marina, universities). This is the standard
  social-security-access definition of informality.
- **`horas_trabajadas`** — weekly hours; values above 168 are treated as not specified.


In [14]:
COLS_TRAB = ID_PERSONA + [
    "id_trabajo", "trapais", "subor", "indep", "personal", "pago", "contrato", "tipocontr",
    "htrab", "sinco", "scian", "clas_emp", "tam_emp", "tipoact", "socios",
] + [f"pres_{i}" for i in range(1, 21)] + [f"medtrab_{i}" for i in range(1, 8)]

trab = leer_enigh("trabajos", usecols=COLS_TRAB)
print(f"trabajos: {len(trab):,} filas")
print(trab["id_trabajo"].value_counts().rename({"1": "principal", "2": "secundario"}).to_string())

principal = trab[trab["id_trabajo"] == "1"].copy()
secundario = trab[trab["id_trabajo"] == "2"][ID_PERSONA].copy()
assert not principal.duplicated(ID_PERSONA).any(), "Hay más de un trabajo principal por persona"

trabajos: 164,325 filas
id_trabajo
principal     150382
secundario     13943


In [15]:
# --------------------------------------------------- Catálogos y mapeos
SINCO_GRUPO = {
    "0": "0 Casos especiales",
    "1": "1 Funcionarios, directores y jefes",
    "2": "2 Profesionistas y técnicos",
    "3": "3 Auxiliares en actividades administrativas",
    "4": "4 Comerciantes, empleados en ventas y agentes",
    "5": "5 Servicios personales y vigilancia",
    "6": "6 Actividades agrícolas, ganaderas, forestales, caza y pesca",
    "7": "7 Artesanales, construcción y otros oficios",
    "8": "8 Operadores de maquinaria, ensambladores y conductores",
    "9": "9 Actividades elementales y de apoyo",
}
SCIAN_SECTOR = {
    "11": "11 Agricultura, cría, forestal, pesca",
    "21": "21 Minería",
    "22": "22 Electricidad, agua y gas",
    "23": "23 Construcción",
    "31": "31-33 Industrias manufactureras", "32": "31-33 Industrias manufactureras",
    "33": "31-33 Industrias manufactureras",
    "43": "43 Comercio al por mayor",
    "46": "46 Comercio al por menor",
    "48": "48-49 Transportes y almacenamiento", "49": "48-49 Transportes y almacenamiento",
    "51": "51 Información en medios masivos",
    "52": "52 Servicios financieros y de seguros",
    "53": "53 Servicios inmobiliarios y de alquiler",
    "54": "54 Servicios profesionales, científicos y técnicos",
    "55": "55 Corporativos",
    "56": "56 Apoyo a negocios y manejo de residuos",
    "61": "61 Servicios educativos",
    "62": "62 Servicios de salud y asistencia social",
    "71": "71 Esparcimiento y recreación",
    "72": "72 Alojamiento temporal y alimentos",
    "81": "81 Otros servicios excepto gobierno",
    "93": "93 Gobierno y organismos internacionales",
}
CLAS_EMP = {
    "1": "Independiente/familiar", "2": "Empresa privada",
    "3": "Institución de gobierno", "4": "Institución no gubernamental",
}
TIPO_ACT = {
    "1": "Industrial", "2": "Comercial", "3": "Servicios", "4": "Agrícola",
    "5": "Cría de animales", "6": "Recolección", "7": "Forestal", "8": "Caza", "9": "Pesca",
}
PRES_COLS = [f"pres_{i}" for i in range(1, 20)]
MED_PUBLICO = [f"medtrab_{i}" for i in range(1, 6)]   # IMSS, ISSSTE, ISSSTE estatal, PEMEX, universidades


def clasificar_posicion(subor, indep, personal, pago):
    if pago in ("2", "3"):
        return "Trabajador sin pago"
    if subor == "1":
        return "Subordinado remunerado"
    if indep == "1":
        return "Empleador" if personal == "1" else "Trabajador por cuenta propia"
    return np.nan

In [16]:
emp = pd.DataFrame(index=principal.index)
emp[ID_PERSONA] = principal[ID_PERSONA]

emp["tiene_trabajo_principal"] = np.int8(1)
emp["posicion_ocupacion"] = pd.Series(
    [clasificar_posicion(s, i, p, g)
     for s, i, p, g in zip(principal["subor"], principal["indep"],
                           principal["personal"], principal["pago"])],
    index=principal.index,
).fillna("No especificado")   # 158 casos sin combinación válida de subor/indep/pago
emp["sinco"] = principal["sinco"]
emp["sinco_grupo"] = principal["sinco"].str[0].map(SINCO_GRUPO)
emp["scian"] = principal["scian"]
emp["scian_sector"] = principal["scian"].str[:2].map(SCIAN_SECTOR).fillna("99 No especificado")

horas = num(principal["htrab"])
emp["horas_trabajadas"] = horas.where(horas.between(1, 168))
emp["tam_empresa"] = num(principal["tam_emp"]).where(lambda s: s <= 11)   # 12 = "no sabe" -> NaN
emp["clase_empleador"] = principal["clas_emp"].map(CLAS_EMP)
emp["tipo_actividad_negocio"] = principal["tipoact"].map(TIPO_ACT)

emp["tiene_contrato_escrito"] = principal["contrato"].map({"1": 1, "2": 0}).astype("Int8")
emp["contrato_indefinido"] = principal["tipocontr"].map({"1": 0, "2": 1}).astype("Int8")
emp["n_prestaciones"] = principal[PRES_COLS].notna().sum(axis=1).astype("int8")
emp["sin_prestaciones"] = principal["pres_20"].notna().astype("int8")
emp["empleo_formal"] = principal[MED_PUBLICO].notna().any(axis=1).astype("int8")
emp["trabaja_en_mexico"] = principal["trapais"].map({"1": 1, "2": 0}).astype("Int8")
emp["tiene_socios"] = principal["socios"].map({"1": 1, "2": 0}).astype("Int8")

emp["tiene_trabajo_secundario"] = (
    emp.set_index(ID_PERSONA).index.isin(secundario.set_index(ID_PERSONA).index).astype("int8")
)

print(f"trabajo principal: {len(emp):,} personas")
display(emp["posicion_ocupacion"].value_counts(dropna=False))
emp.head(3)

trabajo principal: 150,382 personas


posicion_ocupacion
Subordinado remunerado          109046
Trabajador por cuenta propia     20505
Empleador                        13451
Trabajador sin pago               7222
No especificado                    158
Name: count, dtype: int64

,folioviv,foliohog,numren,tiene_trabajo_principal,posicion_ocupacion,sinco,sinco_grupo,scian,scian_sector,horas_trabajadas,tam_empresa,clase_empleador,tipo_actividad_negocio,tiene_contrato_escrito,contrato_indefinido,n_prestaciones,sin_prestaciones,empleo_formal,trabaja_en_mexico,tiene_socios,tiene_trabajo_secundario
0,0100001901,1,01,1,Subordinado remunerado,8352,"8 Operadores de maquinaria, ensambladores y co...",3360,31-33 Industrias manufactureras,48,NaN,Empresa privada,NaN,1,1,16,0,1,1,<NA>,0
1,0100001901,1,02,1,Subordinado remunerado,5211,5 Servicios personales y vigilancia,8121,81 Otros servicios excepto gobierno,45,2.00,Independiente/familiar,NaN,0,<NA>,2,0,0,1,<NA>,0
2,0100001902,1,01,1,Subordinado remunerado,9233,9 Actividades elementales y de apoyo,3360,31-33 Industrias manufactureras,48,11.00,Empresa privada,NaN,1,1,16,0,1,1,<NA>,0


---
## 5. Geography block (`viviendas`)

`ubica_geo` is the 5-digit INEGI geocode: 2 digits of state + 3 of municipality. We split it, attach the
official names from the `ubica_geo` catalog, and add locality size (`tam_loc`) and the socioeconomic stratum
of the dwelling's block (`est_socio`).

`region` is a project-level grouping of the 32 states into 5 macro-regions — it is our convention, not an
INEGI classification, and it exists to give tree models a coarse geographic split that does not blow up into
32 dummies.


In [17]:
REGION = {
    # Noroeste
    "02": "Noroeste", "03": "Noroeste", "25": "Noroeste", "26": "Noroeste",
    # Noreste
    "05": "Noreste", "08": "Noreste", "10": "Noreste", "19": "Noreste", "28": "Noreste", "32": "Noreste",
    # Occidente y Bajío
    "01": "Occidente-Bajío", "06": "Occidente-Bajío", "11": "Occidente-Bajío", "14": "Occidente-Bajío",
    "16": "Occidente-Bajío", "18": "Occidente-Bajío", "24": "Occidente-Bajío",
    # Centro
    "09": "Centro", "13": "Centro", "15": "Centro", "17": "Centro", "21": "Centro", "22": "Centro",
    "29": "Centro",
    # Sur-Sureste
    "04": "Sur-Sureste", "07": "Sur-Sureste", "12": "Sur-Sureste", "20": "Sur-Sureste",
    "23": "Sur-Sureste", "27": "Sur-Sureste", "30": "Sur-Sureste", "31": "Sur-Sureste",
}
assert len(REGION) == 32

TAM_LOC = {"1": "1 >=100,000 hab", "2": "2 15,000-99,999 hab",
           "3": "3 2,500-14,999 hab", "4": "4 <2,500 hab (rural)"}
EST_SOCIO = {"1": "1 Bajo", "2": "2 Medio bajo", "3": "3 Medio alto", "4": "4 Alto"}

In [18]:
COLS_VIV = ID_VIVIENDA + [
    "ubica_geo", "tam_loc", "est_socio", "tipo_viv", "mat_pisos", "num_cuarto", "cuart_dorm",
    "tenencia", "disp_elect", "drenaje", "excusado", "agua_ent", "combus", "tot_resid", "tot_hog",
]
viv = leer_enigh("viviendas", usecols=COLS_VIV)
cat_geo = leer_catalogo("viviendas", "ubica_geo")[["ubica_geo", "desc_ent", "desc_mun"]]

geo = viv[ID_VIVIENDA].copy()
geo["cve_entidad"] = viv["ubica_geo"].str[:2]
geo["cve_municipio"] = viv["ubica_geo"]
geo = geo.merge(cat_geo.rename(columns={"ubica_geo": "cve_municipio",
                                        "desc_ent": "entidad_nombre",
                                        "desc_mun": "municipio_nombre"}),
                on="cve_municipio", how="left")
geo["region"] = geo["cve_entidad"].map(REGION)
geo["tamanio_localidad"] = viv["tam_loc"].map(TAM_LOC)
geo["es_rural"] = (viv["tam_loc"] == "4").astype("int8")
geo["estrato_socioeconomico"] = viv["est_socio"].map(EST_SOCIO)

print(f"viviendas: {len(geo):,} | municipios distintos: {geo['cve_municipio'].nunique():,}")
assert geo["entidad_nombre"].notna().all(), "Municipios sin nombre en el catálogo"
geo.head(3)

viviendas: 90,324 | municipios distintos: 1,112


,folioviv,cve_entidad,cve_municipio,entidad_nombre,municipio_nombre,region,tamanio_localidad,es_rural,estrato_socioeconomico
0,0100001901,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto
1,0100001902,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto
2,0100001904,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto


---
## 6. Household-context block

Three sources, **strictly excluding every income and expenditure variable**:

| Source | Used for |
|---|---|
| `concentradohogar` | composition only — household class, size, age structure, number of employed members, head's sex/age/education |
| `hogares` | assets and connectivity — car, computer, internet, mobile phone, bank card |
| `viviendas` | dwelling quality — type, floor material, rooms, tenure, electricity, drainage, water, crowding |

`ocupados` and `percep_ing` are *counts of people*, not amounts, so they are not a leak of the target —
but they do describe the household's earning capacity, so they are listed explicitly in the data dictionary.


In [19]:
COLS_CON = ID_HOGAR + ["clase_hog", "tot_integ", "hombres", "mujeres", "menores", "mayores",
                        "p12_64", "p65mas", "ocupados", "percep_ing",
                        "sexo_jefe", "edad_jefe", "educa_jefe"]
con = leer_enigh("concentradohogar", usecols=COLS_CON)

CLASE_HOG = {"1": "Unipersonal", "2": "Nuclear", "3": "Ampliado", "4": "Compuesto", "5": "Corresidente"}
EDUCA_JEFE = mapa_catalogo("concentradohogar", "educa_jefe")

hogar = con[ID_HOGAR].copy()
hogar["clase_hogar"] = con["clase_hog"].map(CLASE_HOG)
hogar["integrantes_hogar"] = num(con["tot_integ"])
hogar["menores_hogar"] = num(con["menores"])
hogar["adultos_mayores_hogar"] = num(con["p65mas"])
hogar["ocupados_hogar"] = num(con["ocupados"])
hogar["perceptores_hogar"] = num(con["percep_ing"])
hogar["jefe_mujer"] = (con["sexo_jefe"] == "2").astype("int8")
hogar["edad_jefe"] = num(con["edad_jefe"])
hogar["educacion_jefe"] = con["educa_jefe"].map(EDUCA_JEFE)

# Activos y conectividad del hogar
COLS_HOG = ID_HOGAR + ["telefono", "celular", "conex_inte", "tv_paga",
                       "num_auto", "num_compu", "num_lap", "tarjeta"]
hog_raw = leer_enigh("hogares", usecols=COLS_HOG)
activos = hog_raw[ID_HOGAR].copy()
activos["tiene_internet"] = (hog_raw["conex_inte"] == "1").astype("int8")
activos["tiene_celular"] = (hog_raw["celular"] == "1").astype("int8")
activos["tiene_tarjeta_credito"] = (hog_raw["tarjeta"] == "1").astype("int8")
activos["n_automoviles"] = num(hog_raw["num_auto"]).fillna(0)
activos["tiene_computadora"] = (
    (num(hog_raw["num_compu"]).fillna(0) + num(hog_raw["num_lap"]).fillna(0)) > 0).astype("int8")

hogar = hogar.merge(activos, on=ID_HOGAR, how="left")
print(f"hogares: {len(hogar):,}")
hogar.head(3)

hogares: 91,414


,folioviv,foliohog,clase_hogar,integrantes_hogar,menores_hogar,adultos_mayores_hogar,ocupados_hogar,perceptores_hogar,jefe_mujer,edad_jefe,educacion_jefe,tiene_internet,tiene_celular,tiene_tarjeta_credito,n_automoviles,tiene_computadora
0,0100001901,1,Nuclear,4,2,0,2,2,0,32,Secundaria completa,1,1,0,1,1
1,0100001902,1,Nuclear,4,0,0,2,2,0,48,Profesional incompleta,1,1,1,1,1
2,0100001904,1,Nuclear,2,0,0,2,2,1,60,Secundaria completa,1,1,0,1,0


In [20]:
TIPO_VIV = mapa_catalogo("viviendas", "tipo_viv")
MAT_PISOS = mapa_catalogo("viviendas", "mat_pisos")
TENENCIA = mapa_catalogo("viviendas", "tenencia")

vivienda = viv[ID_VIVIENDA].copy()
vivienda["tipo_vivienda"] = viv["tipo_viv"].map(TIPO_VIV)
vivienda["material_piso"] = viv["mat_pisos"].map(MAT_PISOS)
vivienda["num_cuartos"] = num(viv["num_cuarto"])
vivienda["num_dormitorios"] = num(viv["cuart_dorm"])
vivienda["tenencia_vivienda"] = viv["tenencia"].map(TENENCIA)
vivienda["tiene_electricidad"] = (viv["disp_elect"] != "5").astype("int8")
vivienda["drenaje_publico"] = (viv["drenaje"] == "1").astype("int8")
vivienda["agua_entubada_dentro"] = (viv["agua_ent"] == "1").astype("int8")
vivienda["residentes_vivienda"] = num(viv["tot_resid"])
vivienda["hacinamiento"] = (num(viv["tot_resid"]) / num(viv["num_cuarto"])).round(3)

vivienda.head(3)

,folioviv,tipo_vivienda,material_piso,num_cuartos,num_dormitorios,tenencia_vivienda,tiene_electricidad,drenaje_publico,agua_entubada_dentro,residentes_vivienda,hacinamiento
0,0100001901,Local no construido para habitación,"Madera, mosaico u otro recubrimiento",4,1,Es rentada,1,1,0,4,1.00
1,0100001902,Casa única en el terreno,"Madera, mosaico u otro recubrimiento",4,3,Es propia pero la están pagando,1,1,1,4,1.00
2,0100001904,Casa única en el terreno,"Madera, mosaico u otro recubrimiento",3,2,Es rentada,1,1,1,2,0.67


---
## 7. Join

`poblacion` is the spine: every person exists there. The other blocks are attached with **left joins**, so
no row is created or lost. After the join:

- people with no record in `ingresos` get `ingreso_mensual = 0` (they had no monetary income, which is a
  valid value, not a missing one);
- people with no main job keep `NaN` in the job block, and their **categorical** job columns are then set to
  the explicit category `"NO_APLICA"`.


In [21]:
COLS_TARGET = [c for c in target.columns if c not in ID_PERSONA]
COLS_EMPLEO = [c for c in emp.columns if c not in ID_PERSONA]

dataset = (
    personas
    .merge(target, on=ID_PERSONA, how="left")
    .merge(emp, on=ID_PERSONA, how="left")
    .merge(hogar, on=ID_HOGAR, how="left")
    .merge(geo, on=ID_VIVIENDA, how="left")
    .merge(vivienda, on=ID_VIVIENDA, how="left")
)
assert len(dataset) == len(personas), "El join cambió el número de filas"

# Sin registro en `ingresos` = sin ingreso monetario, no es un faltante
dataset[COLS_TARGET] = dataset[COLS_TARGET].fillna(0.0)

# Bloque laboral: NaN estructural (la persona no trabaja) -> categoría explícita
dataset["tiene_trabajo_principal"] = dataset["tiene_trabajo_principal"].fillna(0).astype("int8")
dataset["tiene_trabajo_secundario"] = dataset["tiene_trabajo_secundario"].fillna(0).astype("int8")

CAT_EMPLEO = ["posicion_ocupacion", "sinco", "sinco_grupo", "scian", "scian_sector",
              "clase_empleador", "tipo_actividad_negocio"]
for col in CAT_EMPLEO:
    dataset[col] = dataset[col].fillna("NO_APLICA")

# El left join convierte los enteros del bloque laboral en float; se restauran como enteros nulables
BIN_EMPLEO = ["n_prestaciones", "sin_prestaciones", "empleo_formal",
              "tiene_contrato_escrito", "contrato_indefinido", "trabaja_en_mexico", "tiene_socios"]
for col in BIN_EMPLEO:
    dataset[col] = dataset[col].astype("Int8")

print("sin trabajo principal:", int((dataset['tiene_trabajo_principal'] == 0).sum()),
      "-> NO_APLICA en las categóricas laborales")

print(f"dataset completo: {len(dataset):,} filas × {dataset.shape[1]} columnas")
dataset.head(3)

sin trabajo principal: 158216 -> NO_APLICA en las categóricas laborales
dataset completo: 308,598 filas × 88 columnas


,folioviv,foliohog,numren,entidad,est_dis,upm,factor,edad,sexo,es_mujer,parentesco_grupo,es_jefe_hogar,situacion_conyugal,vive_en_pareja,lugar_nacimiento,es_migrante_interno,hijos_sobrevivientes,habla_lengua_indigena,se_considera_indigena,es_afrodescendiente,nivel_educativo,anios_escolaridad,sabe_leer_escribir,asiste_a_escuela,n_dificultades,tiene_discapacidad,trabajo_mes_pasado,n_trabajos,actividad_no_ocupado,cotiza_seguridad_social,ing_mens_sueldos_salarios,ing_mens_negocios,ing_mens_trabajo_menores,ing_mens_rentas,ing_mens_transferencias,ing_mens_otros,ingreso_mensual,ingreso_mensual_laboral,tiene_trabajo_principal,posicion_ocupacion,sinco,sinco_grupo,scian,scian_sector,horas_trabajadas,tam_empresa,clase_empleador,tipo_actividad_negocio,tiene_contrato_escrito,contrato_indefinido,n_prestaciones,sin_prestaciones,empleo_formal,trabaja_en_mexico,tiene_socios,tiene_trabajo_secundario,clase_hogar,integrantes_hogar,menores_hogar,adultos_mayores_hogar,ocupados_hogar,perceptores_hogar,jefe_mujer,edad_jefe,educacion_jefe,tiene_internet,tiene_celular,tiene_tarjeta_credito,n_automoviles,tiene_computadora,cve_entidad,cve_municipio,entidad_nombre,municipio_nombre,region,tamanio_localidad,es_rural,estrato_socioeconomico,tipo_vivienda,material_piso,num_cuartos,num_dormitorios,tenencia_vivienda,tiene_electricidad,drenaje_publico,agua_entubada_dentro,residentes_vivienda,hacinamiento
0,0100001901,1,01,01,001,0000001,207,32,Hombre,0,Jefe(a),1,Casado(a),1,Misma entidad,0,NaN,0,0,0,03 Secundaria,9.00,1,0,0,0,1,1,NaN,1,"26,263.04",0.00,0.00,0.00,0.00,0.00,"26,263.04","26,263.04",1,Subordinado remunerado,8352,"8 Operadores de maquinaria, ensambladores y co...",3360,31-33 Industrias manufactureras,48.00,NaN,Empresa privada,NO_APLICA,1,1,16,0,1,1,<NA>,0,Nuclear,4,2,0,2,2,0,32,Secundaria completa,1,1,0,1,1,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto,Local no construido para habitación,"Madera, mosaico u otro recubrimiento",4,1,Es rentada,1,1,0,4,1.00
1,0100001901,1,02,01,001,0000001,207,24,Mujer,1,Cónyuge,0,Casado(a),1,Misma entidad,0,2.00,0,0,0,04 Media superior,12.00,1,0,0,0,1,1,NaN,0,"10,385.86",0.00,0.00,0.00,0.00,0.00,"10,385.86","10,385.86",1,Subordinado remunerado,5211,5 Servicios personales y vigilancia,8121,81 Otros servicios excepto gobierno,45.00,2.00,Independiente/familiar,NO_APLICA,0,<NA>,2,0,0,1,<NA>,0,Nuclear,4,2,0,2,2,0,32,Secundaria completa,1,1,0,1,1,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto,Local no construido para habitación,"Madera, mosaico u otro recubrimiento",4,1,Es rentada,1,1,0,4,1.00
2,0100001901,1,03,01,001,0000001,207,5,Mujer,1,Hijo(a),0,NaN,0,Misma entidad,0,NaN,0,0,0,01 Sin instrucción,0.00,1,1,0,0,0,0,NaN,<NA>,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,NO_APLICA,NO_APLICA,NO_APLICA,NO_APLICA,NO_APLICA,NaN,NaN,NO_APLICA,NO_APLICA,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,Nuclear,4,2,0,2,2,0,32,Secundaria completa,1,1,0,1,1,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto,Local no construido para habitación,"Madera, mosaico u otro recubrimiento",4,1,Es rentada,1,1,0,4,1.00


---
## 8. Analysis universe

Three flags are computed on the full table, then the chosen universe is applied:

| Flag | Definition |
|---|---|
| `es_adulto` | `edad >= 18` |
| `tiene_ingreso` | `ingreso_mensual > 0` |
| `ocupado` | worked last month (`trabajo_mp == 1`) |

`SOLO_INGRESO_POSITIVO = True` keeps **adults 18+ with positive monetary income**. Flip the switch in §1 to
build the other two universes from the same code.


In [22]:
dataset["es_adulto"] = (dataset["edad"] >= EDAD_MINIMA).astype("int8")
dataset["tiene_ingreso"] = (dataset["ingreso_mensual"] > 0).astype("int8")
dataset["ocupado"] = dataset["trabajo_mes_pasado"].astype("int8")

universos = pd.DataFrame({
    "personas": [
        len(dataset),
        int(dataset["es_adulto"].sum()),
        int((dataset["es_adulto"] & dataset["tiene_ingreso"]).sum()),
        int((dataset["es_adulto"] & dataset["ocupado"]).sum()),
    ],
    "poblacion_expandida": [
        dataset["factor"].sum(),
        dataset.loc[dataset["es_adulto"] == 1, "factor"].sum(),
        dataset.loc[(dataset["es_adulto"] == 1) & (dataset["tiene_ingreso"] == 1), "factor"].sum(),
        dataset.loc[(dataset["es_adulto"] == 1) & (dataset["ocupado"] == 1), "factor"].sum(),
    ],
}, index=["Todas las personas", f"Adultos {EDAD_MINIMA}+",
          f"Adultos {EDAD_MINIMA}+ con ingreso>0", f"Adultos {EDAD_MINIMA}+ ocupados"])
universos["poblacion_expandida"] = universos["poblacion_expandida"].round(0)
universos

,personas,poblacion_expandida
Todas las personas,308598,130325969
Adultos 18+,218828,94192177
Adultos 18+ con ingreso>0,179557,77144542
Adultos 18+ ocupados,142995,61317322


In [24]:
mascara = dataset["es_adulto"] == 1
if SOLO_INGRESO_POSITIVO:
    mascara &= dataset["tiene_ingreso"] == 1

df = dataset.loc[mascara].reset_index(drop=True)

# El logaritmo es la transformación habitual del ingreso (distribución lognormal)
df["log_ingreso_mensual"] = np.log(df["ingreso_mensual"].clip(lower=1))

print(f"dataset final: {len(df):,} filas × {df.shape[1]} columnas")
print(f"población representada: {df['factor'].sum():,.0f} personas")

dataset final: 179,557 filas × 92 columnas
población representada: 77,144,542 personas


---
## 9. Quality checks

Four things are verified:

1. **Uniqueness & integrity** — one row per person, no orphan keys, no duplicated identifiers.
2. **Missingness** — which columns have gaps and whether they are structural.
3. **Target distribution** — weighted quantiles, extreme values, share of zeros.
4. **External validation** — the household aggregate of our person-level monetary income is compared with
   INEGI's own `ing_cor − estim_alqu` in `concentradohogar`, and the weighted quarterly household income is
   compared with the figure INEGI publishes for ENIGH 2024.


In [25]:
# --- 9.1 Integridad
print("filas duplicadas por persona :", df.duplicated(ID_PERSONA).sum())
print("folioviv nulos               :", df["folioviv"].isna().sum())
print("factor nulos o <= 0          :", int((df["factor"].isna() | (df["factor"] <= 0)).sum()))
print("ingreso_mensual nulo         :", int(df["ingreso_mensual"].isna().sum()))
print("ingreso_mensual <= 0         :", int((df["ingreso_mensual"] <= 0).sum()))
print("edad fuera de [18, 120]      :", int((~df["edad"].between(18, 120)).sum()))

filas duplicadas por persona : 0
folioviv nulos               : 0
factor nulos o <= 0          : 0
ingreso_mensual nulo         : 0
ingreso_mensual <= 0         : 0
edad fuera de [18, 120]      : 0


In [26]:
# --- 9.2 Valores faltantes
faltantes = (
    pd.DataFrame({"n_nulos": df.isna().sum(), "pct_nulos": (100 * df.isna().mean()).round(2),
                  "dtype": df.dtypes.astype(str)})
    .query("n_nulos > 0")
    .sort_values("pct_nulos", ascending=False)
)
print(f"{len(faltantes)} columnas con faltantes de {df.shape[1]}")
faltantes

13 columnas con faltantes de 92


,n_nulos,pct_nulos,dtype
tiene_socios,150448,83.79,Int8
actividad_no_ocupado,139171,77.51,str
contrato_indefinido,127780,71.16,Int8
hijos_sobrevivientes,112926,62.89,float64
tiene_contrato_escrito,75062,41.80,Int8
tam_empresa,43488,24.22,float64
horas_trabajadas,40386,22.49,float64
sin_prestaciones,40386,22.49,Int8
n_prestaciones,40386,22.49,Int8
empleo_formal,40386,22.49,Int8


In [27]:
# --- 9.3 Distribución del target
peso = df["factor"].to_numpy()
ingreso = df["ingreso_mensual"].to_numpy()

resumen_target = pd.Series({
    "n": len(df),
    "población expandida": peso.sum(),
    "media ponderada": np.average(ingreso, weights=peso),
    "media simple": ingreso.mean(),
    "desv. est.": ingreso.std(),
    "mínimo": ingreso.min(),
    "máximo": ingreso.max(),
})
display(resumen_target.round(2))

q = cuantiles_ponderados(ingreso, peso, qs=(0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999))
display(pd.Series(q, name="ingreso mensual (MXN, ponderado)").round(1))

print("\nComposición del ingreso (participación en el total ponderado):")
comp = {c: np.average(df[c], weights=peso) for c in df.columns if c.startswith("ing_mens_")}
comp = pd.Series(comp).sort_values(ascending=False)
display((100 * comp / comp.sum()).round(2).rename("% del ingreso total"))

n                        179,557.00
población expandida   77,144,542.00
media ponderada           10,677.64
media simple              10,044.39
desv. est.                19,934.85
mínimo                         0.32
máximo                 5,791,304.34
dtype: float64

p1          195.60
p5          896.70
p10       1,614.10
p25       3,913.00
p50       7,907.60
p75      12,962.00
p90      20,869.60
p95      29,250.50
p99      55,543.00
p99.9   138,395.60
Name: ingreso mensual (MXN, ponderado), dtype: float64


Composición del ingreso (participación en el total ponderado):


ing_mens_sueldos_salarios   72.43
ing_mens_transferencias     16.76
ing_mens_negocios            9.50
ing_mens_rentas              1.22
ing_mens_otros               0.10
ing_mens_trabajo_menores     0.00
Name: % del ingreso total, dtype: float64

In [28]:
# --- 9.4 Outliers (marcados, NO eliminados)
umbral_alto = cuantiles_ponderados(ingreso, peso, qs=(0.999,))["p99.9"]
df["outlier_ingreso"] = (df["ingreso_mensual"] > umbral_alto).astype("int8")

print(f"umbral p99.9 ponderado: ${umbral_alto:,.0f} / mes")
print(f"marcados como outlier : {int(df['outlier_ingreso'].sum()):,} ({df['outlier_ingreso'].mean():.2%})")
display(df.nlargest(8, "ingreso_mensual")[
    ["edad", "sexo", "anios_escolaridad", "posicion_ocupacion", "sinco_grupo",
     "entidad_nombre", "ingreso_mensual"]])
print("\nSe marcan, no se eliminan: la decisión de recortar o winsorizar pertenece al notebook de modelado.")

umbral p99.9 ponderado: $138,396 / mes
marcados como outlier : 167 (0.09%)


,edad,sexo,anios_escolaridad,posicion_ocupacion,sinco_grupo,entidad_nombre,ingreso_mensual
106055,62,Hombre,16.00,Empleador,"1 Funcionarios, directores y jefes",Nuevo León,"5,791,304.34"
47164,41,Hombre,0.00,Empleador,"6 Actividades agrícolas, ganaderas, forestales...",Chihuahua,"1,992,827.21"
90937,60,Hombre,16.00,Empleador,"1 Funcionarios, directores y jefes",Michoacán de Ocampo,"1,573,770.48"
47150,30,Hombre,9.00,Empleador,"6 Actividades agrícolas, ganaderas, forestales...",Chihuahua,"1,278,015.65"
103836,66,Hombre,17.00,Empleador,"1 Funcionarios, directores y jefes",Nuevo León,"1,048,351.64"
107732,52,Hombre,17.00,Empleador,"1 Funcionarios, directores y jefes",Nuevo León,"929,168.47"
4238,28,Mujer,17.00,Empleador,"4 Comerciantes, empleados en ventas y agentes",Aguascalientes,"737,327.87"
98593,57,Hombre,9.00,Empleador,"6 Actividades agrícolas, ganaderas, forestales...",Morelos,"729,344.26"



Se marcan, no se eliminan: la decisión de recortar o winsorizar pertenece al notebook de modelado.


In [29]:
# --- 9.5 Validación externa contra concentradohogar
val = leer_enigh("concentradohogar", usecols=ID_HOGAR + ["ing_cor", "estim_alqu", "factor"])
for c in ["ing_cor", "estim_alqu", "factor"]:
    val[c] = num(val[c])

# Nuestro ingreso monetario, agregado a nivel hogar y re-expresado a trimestre
calc = (dataset.groupby(ID_HOGAR)["ingreso_mensual"].sum() * 3).rename("monetario_calc")
val = val.merge(calc, on=ID_HOGAR, how="left")
val["monetario_calc"] = val["monetario_calc"].fillna(0.0)
val["monetario_inegi"] = val["ing_cor"] - val["estim_alqu"]   # ing_cor sin renta imputada
dif = val["monetario_calc"] - val["monetario_inegi"]

print(f"hogares comparados                        : {len(val):,}")
print(f"coincidencia exacta (|dif| < $1)          : {(dif.abs() < 1).mean():.1%}")
print(f"nuestro cálculo EXCEDE al de INEGI        : {(dif > 1).mean():.3%}  <- debe ser ~0%")
print(f"diferencia total (ingreso en especie)     : ${dif[dif < 0].abs().sum():,.0f} trimestrales")
print("""
Interpretación: nuestro agregado nunca excede al de INEGI y coincide exactamente en ~47% de los hogares.
El faltante es el ingreso NO monetario (pago en especie, autoconsumo, regalos y transferencias en especie),
que la ENIGH sólo mide a nivel hogar y por tanto no puede asignarse a una persona. Confirma que
`ingreso_mensual` es exactamente el ingreso corriente MONETARIO de la persona.
""")

ing_cor_trim_hogar = np.average(val["ing_cor"], weights=val["factor"])
print(f"Ingreso corriente trimestral promedio por hogar (ponderado): ${ing_cor_trim_hogar:,.0f}")
print("Cifra publicada por INEGI para la ENIGH 2024               : $77,864")

hogares comparados                        : 91,414
coincidencia exacta (|dif| < $1)          : 46.6%
nuestro cálculo EXCEDE al de INEGI        : 0.028%  <- debe ser ~0%
diferencia total (ingreso en especie)     : $383,514,963 trimestrales

Interpretación: nuestro agregado nunca excede al de INEGI y coincide exactamente en ~47% de los hogares.
El faltante es el ingreso NO monetario (pago en especie, autoconsumo, regalos y transferencias en especie),
que la ENIGH sólo mide a nivel hogar y por tanto no puede asignarse a una persona. Confirma que
`ingreso_mensual` es exactamente el ingreso corriente MONETARIO de la persona.

Ingreso corriente trimestral promedio por hogar (ponderado): $77,864
Cifra publicada por INEGI para la ENIGH 2024               : $77,864


---
## 10. Save the dataset and its data dictionary

Outputs in `data/processed/`:

| File | Content |
|---|---|
| `enigh2024_ingreso_personas.parquet` | the final dataset (preferred format: typed, compressed) |
| `enigh2024_ingreso_personas.csv` | same content, plain text |
| `diccionario_enigh2024_ingreso_personas.csv` | one row per column: block, description, type, % missing |
| `metadata_enigh2024_ingreso_personas.json` | build parameters and row counts, for reproducibility |


In [30]:
DICCIONARIO = {
    # --- identificadores y diseño muestral
    "folioviv": ("Identificador", "Identificador de la vivienda"),
    "foliohog": ("Identificador", "Identificador del hogar dentro de la vivienda"),
    "numren": ("Identificador", "Número de renglón (persona) dentro del hogar"),
    "factor": ("Diseño muestral", "Factor de expansión. NO es variable predictora"),
    "upm": ("Diseño muestral", "Unidad primaria de muestreo (conglomerado). Usar para agrupar en el split"),
    "est_dis": ("Diseño muestral", "Estrato de diseño muestral"),
    "entidad": ("Diseño muestral", "Clave de entidad federativa (tabla poblacion)"),
    # --- target
    "ingreso_mensual": ("TARGET", "Ingreso corriente MONETARIO mensual de la persona, en pesos de ago-2024"),
    "log_ingreso_mensual": ("TARGET", "Logaritmo natural de ingreso_mensual"),
    "ingreso_mensual_laboral": ("TARGET alternativo", "Ingreso mensual por trabajo (sueldos + negocios + menores)"),
    "ing_mens_sueldos_salarios": ("Componente", "P001-P022: sueldos, destajo, horas extra, comisiones, aguinaldo, utilidades"),
    "ing_mens_negocios": ("Componente", "P068-P081: ganancias de negocios propios por tipo de actividad"),
    "ing_mens_trabajo_menores": ("Componente", "P067: ingreso por trabajo de menores de 12 años"),
    "ing_mens_rentas": ("Componente", "P023-P031: alquileres, intereses, regalías"),
    "ing_mens_transferencias": ("Componente", "P032-P048 y P101-P108: jubilaciones, becas, remesas, programas sociales, donativos"),
    "ing_mens_otros": ("Componente", "P049: otros ingresos corrientes"),
    # --- persona
    "edad": ("Persona", "Edad en años cumplidos"),
    "sexo": ("Persona", "Sexo declarado"),
    "es_mujer": ("Persona", "1 si es mujer"),
    "parentesco_grupo": ("Persona", "Parentesco con la jefatura del hogar, agrupado"),
    "es_jefe_hogar": ("Persona", "1 si es jefe(a) del hogar"),
    "situacion_conyugal": ("Persona", "Situación conyugal"),
    "vive_en_pareja": ("Persona", "1 si vive en pareja (unión libre o casado/a)"),
    "lugar_nacimiento": ("Persona", "Entidad o país de nacimiento"),
    "es_migrante_interno": ("Persona", "1 si nació fuera de la entidad de residencia"),
    "hijos_sobrevivientes": ("Persona", "Número de hijos sobrevivientes (sólo mujeres 12+)"),
    "habla_lengua_indigena": ("Persona", "1 si habla alguna lengua indígena"),
    "se_considera_indigena": ("Persona", "1 si se autoadscribe como indígena"),
    "es_afrodescendiente": ("Persona", "1 si se autoadscribe como afrodescendiente"),
    "n_dificultades": ("Persona", "Número de dominios con mucha dificultad o imposibilidad (0-8)"),
    "tiene_discapacidad": ("Persona", "1 si tiene al menos una dificultad severa"),
    # --- educación
    "nivel_educativo": ("Educación", "Nivel de instrucción aprobado, agrupado en 6 categorías"),
    "anios_escolaridad": ("Educación", "Años de escolaridad reconstruidos (nivelaprob x gradoaprob, criterio CONEVAL)"),
    "sabe_leer_escribir": ("Educación", "1 si es alfabeta"),
    "asiste_a_escuela": ("Educación", "1 si asiste actualmente a la escuela"),
    # --- trabajo
    "trabajo_mes_pasado": ("Trabajo", "1 si trabajó el mes pasado"),
    "ocupado": ("Trabajo", "Igual a trabajo_mes_pasado; bandera de universo"),
    "n_trabajos": ("Trabajo", "Número de trabajos declarados"),
    "actividad_no_ocupado": ("Trabajo", "Actividad principal de quien no trabajó"),
    "cotiza_seguridad_social": ("Trabajo", "1 si contribuye a la seguridad social"),
    "tiene_trabajo_principal": ("Trabajo", "1 si tiene registro de trabajo principal"),
    "tiene_trabajo_secundario": ("Trabajo", "1 si además reporta un trabajo secundario"),
    "posicion_ocupacion": ("Trabajo", "Subordinado / empleador / cuenta propia / sin pago / NO_APLICA"),
    "sinco": ("Trabajo", "Ocupación SINCO-2011 a 4 dígitos"),
    "sinco_grupo": ("Trabajo", "Gran grupo ocupacional SINCO a 1 dígito"),
    "scian": ("Trabajo", "Actividad económica SCIAN a 4 dígitos"),
    "scian_sector": ("Trabajo", "Sector económico SCIAN a 2 dígitos"),
    "horas_trabajadas": ("Trabajo", "Horas trabajadas a la semana en el trabajo principal"),
    "tam_empresa": ("Trabajo", "Tamaño del establecimiento, escala ordinal 1-11"),
    "clase_empleador": ("Trabajo", "Tipo de unidad económica donde trabaja"),
    "tipo_actividad_negocio": ("Trabajo", "Tipo de actividad del negocio (independientes)"),
    "tiene_contrato_escrito": ("Trabajo", "1 si tiene contrato por escrito"),
    "contrato_indefinido": ("Trabajo", "1 si el contrato es de base o por tiempo indeterminado"),
    "n_prestaciones": ("Trabajo", "Número de prestaciones laborales (0-19)"),
    "sin_prestaciones": ("Trabajo", "1 si declaró no tener prestaciones"),
    "empleo_formal": ("Trabajo", "Proxy de formalidad: el trabajo da acceso a una institución pública de salud"),
    "trabaja_en_mexico": ("Trabajo", "1 si el trabajo se realiza en el país"),
    "tiene_socios": ("Trabajo", "1 si el negocio tiene socios"),
    # --- geografía
    "cve_entidad": ("Geografía", "Clave INEGI de entidad federativa (2 dígitos)"),
    "entidad_nombre": ("Geografía", "Nombre de la entidad federativa"),
    "cve_municipio": ("Geografía", "Clave INEGI de municipio (5 dígitos)"),
    "municipio_nombre": ("Geografía", "Nombre del municipio"),
    "region": ("Geografía", "Macro-región (convención del proyecto, 5 clases)"),
    "tamanio_localidad": ("Geografía", "Tamaño de la localidad"),
    "es_rural": ("Geografía", "1 si la localidad tiene menos de 2,500 habitantes"),
    "estrato_socioeconomico": ("Geografía", "Estrato socioeconómico de la manzana"),
    # --- hogar
    "clase_hogar": ("Hogar", "Clase de hogar"),
    "integrantes_hogar": ("Hogar", "Total de integrantes del hogar"),
    "menores_hogar": ("Hogar", "Integrantes menores de 12 años"),
    "adultos_mayores_hogar": ("Hogar", "Integrantes de 65 años o más"),
    "ocupados_hogar": ("Hogar", "Integrantes ocupados"),
    "perceptores_hogar": ("Hogar", "Integrantes con ingreso (conteo, no monto)"),
    "jefe_mujer": ("Hogar", "1 si la jefatura del hogar es femenina"),
    "edad_jefe": ("Hogar", "Edad de la jefatura del hogar"),
    "educacion_jefe": ("Hogar", "Nivel educativo de la jefatura del hogar"),
    "tiene_internet": ("Hogar (activos)", "1 si el hogar tiene conexión a internet"),
    "tiene_celular": ("Hogar (activos)", "1 si el hogar tiene teléfono celular"),
    "tiene_tarjeta_credito": ("Hogar (activos)", "1 si algún integrante tiene tarjeta de crédito"),
    "n_automoviles": ("Hogar (activos)", "Número de automóviles"),
    "tiene_computadora": ("Hogar (activos)", "1 si el hogar tiene computadora o laptop"),
    # --- vivienda
    "tipo_vivienda": ("Vivienda", "Tipo de vivienda"),
    "material_piso": ("Vivienda", "Material predominante en pisos"),
    "num_cuartos": ("Vivienda", "Número de cuartos"),
    "num_dormitorios": ("Vivienda", "Número de cuartos usados como dormitorio"),
    "tenencia_vivienda": ("Vivienda", "Tenencia de la vivienda"),
    "tiene_electricidad": ("Vivienda", "1 si dispone de energía eléctrica"),
    "drenaje_publico": ("Vivienda", "1 si tiene drenaje conectado a la red pública"),
    "agua_entubada_dentro": ("Vivienda", "1 si tiene agua entubada dentro de la vivienda"),
    "residentes_vivienda": ("Vivienda", "Total de residentes de la vivienda"),
    "hacinamiento": ("Vivienda", "Residentes por cuarto"),
    # --- banderas
    "es_adulto": ("Bandera", f"1 si edad >= {EDAD_MINIMA}"),
    "tiene_ingreso": ("Bandera", "1 si ingreso_mensual > 0"),
    "outlier_ingreso": ("Bandera", "1 si el ingreso supera el percentil 99.9 ponderado"),
}
print(f"{len(DICCIONARIO)} columnas documentadas")

93 columnas documentadas


In [31]:
# Se descartan los activos del hogar si el switch está apagado
if not INCLUIR_ACTIVOS_HOGAR:
    activos_cols = [c for c, (bloque, _) in DICCIONARIO.items() if bloque == "Hogar (activos)"]
    df = df.drop(columns=[c for c in activos_cols if c in df.columns])
    print("Activos del hogar descartados:", activos_cols)

# Toda columna del dataset debe estar documentada, y viceversa
sin_documentar = [c for c in df.columns if c not in DICCIONARIO]
documentadas_ausentes = [c for c in DICCIONARIO if c not in df.columns]
assert not sin_documentar, f"Columnas sin documentar: {sin_documentar}"
if documentadas_ausentes:
    print("Documentadas pero ausentes (esperado si se apagó algún switch):", documentadas_ausentes)

diccionario = pd.DataFrame(
    [{"columna": c,
      "bloque": DICCIONARIO[c][0],
      "descripcion": DICCIONARIO[c][1],
      "tipo": str(df[c].dtype),
      "pct_nulos": round(100 * df[c].isna().mean(), 2),
      "n_categorias": int(df[c].nunique()) if df[c].dtype == object else np.nan}
     for c in df.columns]
)
diccionario

,columna,bloque,descripcion,tipo,pct_nulos,n_categorias
0,folioviv,Identificador,Identificador de la vivienda,str,0.00,NaN
1,foliohog,Identificador,Identificador del hogar dentro de la vivienda,str,0.00,NaN
2,numren,Identificador,Número de renglón (persona) dentro del hogar,str,0.00,NaN
3,entidad,Diseño muestral,Clave de entidad federativa (tabla poblacion),str,0.00,NaN
4,est_dis,Diseño muestral,Estrato de diseño muestral,str,0.00,NaN
...,...,...,...,...,...,...
88,es_adulto,Bandera,1 si edad >= 18,int8,0.00,NaN
89,tiene_ingreso,Bandera,1 si ingreso_mensual > 0,int8,0.00,NaN
90,ocupado,Trabajo,Igual a trabajo_mes_pasado; bandera de universo,int8,0.00,NaN
91,log_ingreso_mensual,TARGET,Logaritmo natural de ingreso_mensual,float64,0.00,NaN


In [32]:
df.head()

,folioviv,foliohog,numren,entidad,est_dis,upm,factor,edad,sexo,es_mujer,parentesco_grupo,es_jefe_hogar,situacion_conyugal,vive_en_pareja,lugar_nacimiento,es_migrante_interno,hijos_sobrevivientes,habla_lengua_indigena,se_considera_indigena,es_afrodescendiente,nivel_educativo,anios_escolaridad,sabe_leer_escribir,asiste_a_escuela,n_dificultades,tiene_discapacidad,trabajo_mes_pasado,n_trabajos,actividad_no_ocupado,cotiza_seguridad_social,ing_mens_sueldos_salarios,ing_mens_negocios,ing_mens_trabajo_menores,ing_mens_rentas,ing_mens_transferencias,ing_mens_otros,ingreso_mensual,ingreso_mensual_laboral,tiene_trabajo_principal,posicion_ocupacion,sinco,sinco_grupo,scian,scian_sector,horas_trabajadas,tam_empresa,clase_empleador,tipo_actividad_negocio,tiene_contrato_escrito,contrato_indefinido,n_prestaciones,sin_prestaciones,empleo_formal,trabaja_en_mexico,tiene_socios,tiene_trabajo_secundario,clase_hogar,integrantes_hogar,menores_hogar,adultos_mayores_hogar,ocupados_hogar,perceptores_hogar,jefe_mujer,edad_jefe,educacion_jefe,tiene_internet,tiene_celular,tiene_tarjeta_credito,n_automoviles,tiene_computadora,cve_entidad,cve_municipio,entidad_nombre,municipio_nombre,region,tamanio_localidad,es_rural,estrato_socioeconomico,tipo_vivienda,material_piso,num_cuartos,num_dormitorios,tenencia_vivienda,tiene_electricidad,drenaje_publico,agua_entubada_dentro,residentes_vivienda,hacinamiento,es_adulto,tiene_ingreso,ocupado,log_ingreso_mensual,outlier_ingreso
0,0100001901,1,01,01,001,0000001,207,32,Hombre,0,Jefe(a),1,Casado(a),1,Misma entidad,0,NaN,0,0,0,03 Secundaria,9.00,1,0,0,0,1,1,NaN,1,"26,263.04",0.00,0.00,0.00,0.00,0.00,"26,263.04","26,263.04",1,Subordinado remunerado,8352,"8 Operadores de maquinaria, ensambladores y co...",3360,31-33 Industrias manufactureras,48.00,NaN,Empresa privada,NO_APLICA,1,1,16,0,1,1,<NA>,0,Nuclear,4,2,0,2,2,0,32,Secundaria completa,1,1,0,1,1,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto,Local no construido para habitación,"Madera, mosaico u otro recubrimiento",4,1,Es rentada,1,1,0,4,1.00,1,1,1,10.18,0
1,0100001901,1,02,01,001,0000001,207,24,Mujer,1,Cónyuge,0,Casado(a),1,Misma entidad,0,2.00,0,0,0,04 Media superior,12.00,1,0,0,0,1,1,NaN,0,"10,385.86",0.00,0.00,0.00,0.00,0.00,"10,385.86","10,385.86",1,Subordinado remunerado,5211,5 Servicios personales y vigilancia,8121,81 Otros servicios excepto gobierno,45.00,2.00,Independiente/familiar,NO_APLICA,0,<NA>,2,0,0,1,<NA>,0,Nuclear,4,2,0,2,2,0,32,Secundaria completa,1,1,0,1,1,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto,Local no construido para habitación,"Madera, mosaico u otro recubrimiento",4,1,Es rentada,1,1,0,4,1.00,1,1,1,9.25,0
2,0100001902,1,01,01,001,0000001,207,48,Hombre,0,Jefe(a),1,Unión libre,1,Otra entidad,1,NaN,0,0,0,05 Licenciatura,15.00,1,0,0,0,1,1,NaN,1,"18,913.04",0.00,0.00,0.00,0.00,0.00,"18,913.04","18,913.04",1,Subordinado remunerado,9233,9 Actividades elementales y de apoyo,3360,31-33 Industrias manufactureras,48.00,11.00,Empresa privada,NO_APLICA,1,1,16,0,1,1,<NA>,0,Nuclear,4,0,0,2,2,0,48,Profesional incompleta,1,1,1,1,1,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto,Casa única en el terreno,"Madera, mosaico u otro recubrimiento",4,3,Es propia pero la están pagando,1,1,1,4,1.00,1,1,1,9.85,0
3,0100001902,1,03,01,001,0000001,207,22,Hombre,0,Hijo(a),0,Soltero(a),0,Misma entidad,0,NaN,0,0,0,05 Licenciatura,16.00,1,0,0,0,1,1,NaN,1,"9,782.61",0.00,0.00,0.00,0.00,0.00,"9,782.61","9,782.61",1,Subordinado remunerado,9232,9 Actividades elementales y de apoyo,3360,31-33 Industrias manufactureras,48.00,11.00,Empresa privada,NO_APLICA,1,0,11,0,1,1,<NA>,0,Nuclear,4,0,0,2,2,0,48,Profesional incompleta,1,1,1,1,1,01,01001,Aguascalientes,Aguascalientes,Occidente-Bajío,"1 >=100,000 hab",0,3 Medio alto,Casa única en el terreno,"Madera, mosaico u otro recubrimiento",4,3,Es propia pero la están pagando,1,1,1,4,1.00,1,1,1,9.19,0
4,0100001904,1,01,01,001

In [34]:
df['ingreso_mensual'].describe().round(2)

count     179,557.00
mean       10,044.39
std        19,934.91
min             0.32
25%         3,508.20
50%         7,679.34
75%        12,440.21
max     5,791,304.34
Name: ingreso_mensual, dtype: float64

In [33]:
# Orden final: identificadores -> diseño -> target -> predictoras -> banderas
ORDEN_BLOQUES = ["Identificador", "Diseño muestral", "TARGET", "TARGET alternativo", "Componente",
                 "Persona", "Educación", "Trabajo", "Geografía", "Hogar", "Hogar (activos)",
                 "Vivienda", "Bandera"]
orden = sorted(df.columns, key=lambda c: (ORDEN_BLOQUES.index(DICCIONARIO[c][0]), c))
df = df[orden]
diccionario = diccionario.set_index("columna").loc[orden].reset_index()

ruta_parquet = PROC_DIR / f"{NOMBRE_SALIDA}.parquet"
ruta_csv = PROC_DIR / f"{NOMBRE_SALIDA}.csv"
ruta_dicc = PROC_DIR / f"diccionario_{NOMBRE_SALIDA}.csv"
ruta_meta = PROC_DIR / f"metadata_{NOMBRE_SALIDA}.json"

try:
    df.to_parquet(ruta_parquet, index=False)
    print(f"OK  {ruta_parquet.name}  ({ruta_parquet.stat().st_size / 1e6:.1f} MB)")
except ImportError:
    print("pyarrow no está instalado -> se omite el parquet. `pip install pyarrow`")

if GUARDAR_CSV:
    df.to_csv(ruta_csv, index=False, encoding="utf-8")
    print(f"OK  {ruta_csv.name}  ({ruta_csv.stat().st_size / 1e6:.1f} MB)")

diccionario.to_csv(ruta_dicc, index=False, encoding="utf-8")
print(f"OK  {ruta_dicc.name}")

metadata = {
    "fuente": "INEGI - ENIGH 2024, Nueva Serie (microdatos CSV)",
    "target": "ingreso_mensual = ingreso corriente monetario trimestral / 3, pesos de agosto 2024",
    "claves_excluidas": "P050-P066 (percepciones financieras y de capital)",
    "universo": f"personas de {EDAD_MINIMA} años o más"
                + (" con ingreso monetario > 0" if SOLO_INGRESO_POSITIVO else ""),
    "parametros": {"EDAD_MINIMA": EDAD_MINIMA,
                   "SOLO_INGRESO_POSITIVO": SOLO_INGRESO_POSITIVO,
                   "INCLUIR_ACTIVOS_HOGAR": INCLUIR_ACTIVOS_HOGAR},
    "filas": int(len(df)),
    "columnas": int(df.shape[1]),
    "poblacion_expandida": float(df["factor"].sum()),
    "ingreso_mensual_media_ponderada": float(np.average(df["ingreso_mensual"], weights=df["factor"])),
    "ingreso_mensual_mediana_ponderada": float(cuantiles_ponderados(
        df["ingreso_mensual"], df["factor"], qs=(0.5,))["p50"]),
    "advertencias": [
        "factor, upm y est_dis son variables de diseño muestral: no usarlas como predictoras.",
        "Agrupar por upm al separar train/test para no romper los conglomerados.",
        "Los outliers están marcados (outlier_ingreso), no eliminados.",
    ],
}
ruta_meta.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"OK  {ruta_meta.name}")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

OK  enigh2024_ingreso_personas.parquet  (7.4 MB)
OK  enigh2024_ingreso_personas.csv  (101.2 MB)
OK  diccionario_enigh2024_ingreso_personas.csv
OK  metadata_enigh2024_ingreso_personas.json
{
  "fuente": "INEGI - ENIGH 2024, Nueva Serie (microdatos CSV)",
  "target": "ingreso_mensual = ingreso corriente monetario trimestral / 3, pesos de agosto 2024",
  "claves_excluidas": "P050-P066 (percepciones financieras y de capital)",
  "universo": "personas de 18 años o más con ingreso monetario > 0",
  "parametros": {
    "EDAD_MINIMA": 18,
    "SOLO_INGRESO_POSITIVO": true,
    "INCLUIR_ACTIVOS_HOGAR": true
  },
  "filas": 179557,
  "columnas": 93,
  "poblacion_expandida": 77144542.0,
  "ingreso_mensual_media_ponderada": 10677.636890786778,
  "ingreso_mensual_mediana_ponderada": 7907.61,
  "advertencias": [
    "factor, upm y est_dis son variables de diseño muestral: no usarlas como predictoras.",
    "Agrupar por upm al separar train/test para no romper los conglomerados.",
    "Los outliers e

---
## Summary

`data/processed/enigh2024_ingreso_personas.parquet` holds one row per adult (18+) with positive monetary
income, their monthly income in August-2024 pesos, and features across five blocks: person, education, job,
geography and household/dwelling.

### Before modelling

1. **Weights.** Every population statistic must use `factor`. For the model itself, decide explicitly whether
   to fit weighted (estimates the population relationship) or unweighted (estimates the sample relationship);
   with a survey this size, weighted is usually the honest choice for anything reported as a national number.
2. **Split by `upm`.** ENIGH is a clustered sample. A random row-level split leaks, because neighbours in the
   same primary sampling unit are highly correlated. Use `GroupShuffleSplit(groups=df["upm"])`.
3. **Target transform.** Income is close to lognormal; `log_ingreso_mensual` is already there. Remember that
   the mean of the logs is not the log of the mean — back-transform with a smearing estimator if you need
   pesos.
4. **High-cardinality categoricals.** `sinco` (≈480 codes), `scian` (≈177) and `cve_municipio` (≈1,100) need
   target/ordinal encoding or a gradient-boosting model with native categorical support; `sinco_grupo`,
   `scian_sector` and `region` are the low-cardinality fallbacks.
5. **Leakage watch.** `ing_mens_*` and `ingreso_mensual_laboral` are *components of the target* — drop them
   from `X`. So are `ocupados_hogar` and `perceptores_hogar` if you want a strictly individual model.
6. **`NO_APLICA`.** In the job block this is a real category (the person does not work), not a missing value.
   Keep it as a level; do not impute it.
